In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [2]:
from util import *
import os
import torch.nn as nn
import numpy as np

# Train and Test

In [3]:
from cnn import model, input
from cnn.trainer import BaseTrainer  # assuming the code above is saved in trainer.py
from cnn.input import GenericDataLoader
from util import init_log
import copy
import qnas_config as cfg

In [4]:
LOGGER = init_log("INFO", name=__name__)

In [13]:
phase = 'evolution'
experiment_path = 'my_exp_config3'
config_file = 'config_files_cifar/config0.txt'

args = {
    'experiment_path': experiment_path,
    'config_file': config_file,
    'data_path': 'cifar10_data',
    'log_level': 'INFO',
    'log_level': 'INFO',
    'fitness_metric': 'best_accuracy',
    'optimizer': 'adamw',
    'data_augmentation': 'False',
    'dataset': 'cifar10',
    'save_checkpoints_epochs': 10,
    'early_stopping': 'False',
    'en_pop_crossover': 'False',
    'network_config': 'default',
    'network_gap': 'False',
    'limit_data_value': 10000,
    'batch_size': 64,
    'max_epochs': 5,
    'epochs_to_eval': 5,
    'mixed_precision': True,
    "phase": 'evolution',
    'mo_metric_base': 'loss',
    'max_params': 114090,
    'max_inference_time': 1000,
    'task': 'multi-class',

    
}

config = cfg.ConfigParameters(args, phase=phase)
config.get_parameters()

fn_dict=config.fn_dict
fn_dict_tf = copy.deepcopy(fn_dict)
fn_dict

net_list = ['conv_3_1_256','no_op', 'conv_3_1_128', 'no_op','no_op','no_op',
            'conv_3_1_64','conv_3_1_64','no_op','conv_3_1_256','conv_3_1_256',
            'max_pool_2_2', 'conv_3_1_128', 'no_op','no_op','no_op','no_op','no_op',
            'max_pool_2_2', 'conv_3_1_128']

params = config.train_spec

In [14]:
data_loader = GenericDataLoader(params=params)
train_loader, val_loader = data_loader.get_loader(pin_memory_device='cuda:0')

In [15]:
test_loader= data_loader.get_loader(pin_memory_device='cuda:0', for_train=False)

In [16]:
if args['dataset'].lower() in input.available_datasets:
    dataset_info = input.available_datasets[args['dataset'].lower()]
else:
    dataset_info = load_yaml(os.path.join(args['data_path'], 'data_info.txt'))

args['num_classes'] = dataset_info['num_classes']
args['task'] = dataset_info['task']
args['device'] = 'cuda:0'
args['fn_dict'] = fn_dict
args['net_list'] = net_list

In [17]:
has_cbam_key = any(key.startswith('cbam') for key in fn_dict)

In [18]:
# Create the model    
model_net = model.NetworkGraph(num_classes=dataset_info['num_classes'], 
                                network_config=args['network_config'], 
                                network_gap=args['network_gap'])

filtered_dict = {key: item for key, item in fn_dict.items() if key in net_list}
model_net.create_functions(fn_dict=filtered_dict, net_list=net_list, cbam=has_cbam_key)

# Add the fully connected layer to the model
input_shape =  [args['batch_size']] + dataset_info['shape']
inputs = torch.randn(input_shape)
with torch.no_grad():
    _ = model_net(inputs)

args['input_shape'] = input_shape

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model_net.parameters())

In [19]:
model_path = os.path.join(args['experiment_path'], '10_1')
if not os.path.exists(model_path):
    os.makedirs(model_path)

args['model_path'] = model_path

In [20]:
trainer = BaseTrainer(model_net, criterion, optimizer, train_loader, val_loader, test_loader, args, LOGGER)
results = trainer.train()
print("Training complete. Best Accuracy:", results['best_accuracy'])

INFO: trainer: 2025-03-28 15:33:18.296 - Epoch 1/5: Train Loss: 2.4176 | Train Acc: 21.53%
INFO: trainer: 2025-03-28 15:33:36.160 - Epoch 1/5: Val Loss: 1.9502 | Val Acc: 31.20%
INFO: trainer: 2025-03-28 15:33:55.866 - Epoch 2/5: Train Loss: 1.9195 | Train Acc: 32.74%
INFO: trainer: 2025-03-28 15:34:14.146 - Epoch 2/5: Val Loss: 1.7249 | Val Acc: 40.20%
INFO: trainer: 2025-03-28 15:34:33.814 - Epoch 3/5: Train Loss: 1.8527 | Train Acc: 34.89%
INFO: trainer: 2025-03-28 15:34:51.512 - Epoch 3/5: Val Loss: 1.4861 | Val Acc: 45.70%
INFO: trainer: 2025-03-28 15:35:11.004 - Epoch 4/5: Train Loss: 1.7627 | Train Acc: 38.81%
INFO: trainer: 2025-03-28 15:35:28.306 - Epoch 4/5: Val Loss: 1.5572 | Val Acc: 45.20%
INFO: trainer: 2025-03-28 15:35:47.816 - Epoch 5/5: Train Loss: 1.6462 | Train Acc: 41.48%
INFO: trainer: 2025-03-28 15:36:05.289 - Epoch 5/5: Val Loss: 1.4507 | Val Acc: 48.50%


Training complete. Best Accuracy: 48.5


In [21]:
results

{'training_losses': [2.4175650113158755,
  1.919490658574634,
  1.8527229163381789,
  1.762725508875317,
  1.6462450457943811],
 'training_accuracies': [21.533333333333335,
  32.74444444444445,
  34.888888888888886,
  38.81111111111111,
  41.477777777777774],
 'validation_losses': [1.9501599371433258,
  1.7248561680316925,
  1.4861310422420502,
  1.557152271270752,
  1.4506556689739227],
 'validation_accuracies': [31.2, 40.2, 45.7, 45.2, 48.5],
 'best_accuracy': 48.5,
 'best_epoch': 5,
 'training_time': 186.61134362220764,
 'cuda_inference_time': 1798.0337142944336,
 'total_trainable_params': 1677834,
 'model_memory_usage': 266.24267578125,
 'fitness_val_loss': 40.80540618824247,
 'scalar_multi_objective': 1.834182295833547,
 'total_flops': 2524610560,
 'confusion_matrix': None,
 'auc_med': None,
 'acc_test_med': None,
 'test_accuracy': None,
 'test_loss': None}

# Resnet

In [22]:
from cnn import model_resnet
from cnn.trainer import ResNetTrainer  # Our subclass
import torch.nn as nn
import torch.optim as optim

In [25]:
# Set up your parameters, data loaders, etc.
params = {
    'device': 'cuda',
    'max_epochs': 5,
    'epochs_to_eval': 5,
    'mixed_precision': True,
    'lr_scheduler': 'multistep',
    'model_path': './experiment/resnet18',
    'input_shape': [64, 3, 32, 32],
    'num_classes': 10,
    'dataset': 'cifar10',
    'data_path': './data',
    'phase': 'resnet',
    'fn_dict': {},  # not used for ResNet in this case
    'net_list': [],  # not used for ResNet in this case
    'task': 'classification',
}
# Instantiate your ResNet model (you can use a dummy model here, as it will be reloaded)
model_net = model_resnet.ResNet18(in_channels=3, num_classes=10)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model_net.parameters())
# Assume train_loader, val_loader, test_loader are already defined

trainer = ResNetTrainer(model_net, criterion, optimizer, train_loader, val_loader, test_loader, params)
results = trainer.train()
print("Training complete. Best Accuracy:", results['best_accuracy'])

INFO: trainer: 2025-03-28 15:46:02.353 - Epoch 1/5: Train Loss: 2.1351 | Train Acc: 23.66%
INFO: trainer: 2025-03-28 15:46:02.353 - Epoch 1/5: Train Loss: 2.1351 | Train Acc: 23.66%
INFO: trainer: 2025-03-28 15:46:20.051 - Epoch 1/5: Val Loss: 2.3984 | Val Acc: 29.50%
INFO: trainer: 2025-03-28 15:46:20.051 - Epoch 1/5: Val Loss: 2.3984 | Val Acc: 29.50%
INFO: trainer: 2025-03-28 15:46:38.712 - Epoch 2/5: Train Loss: 1.8881 | Train Acc: 30.24%
INFO: trainer: 2025-03-28 15:46:38.712 - Epoch 2/5: Train Loss: 1.8881 | Train Acc: 30.24%
INFO: trainer: 2025-03-28 15:46:56.237 - Epoch 2/5: Val Loss: 1.7182 | Val Acc: 35.20%
INFO: trainer: 2025-03-28 15:46:56.237 - Epoch 2/5: Val Loss: 1.7182 | Val Acc: 35.20%
INFO: trainer: 2025-03-28 15:47:14.907 - Epoch 3/5: Train Loss: 1.7982 | Train Acc: 34.47%
INFO: trainer: 2025-03-28 15:47:14.907 - Epoch 3/5: Train Loss: 1.7982 | Train Acc: 34.47%
INFO: trainer: 2025-03-28 15:47:32.228 - Epoch 3/5: Val Loss: 1.6010 | Val Acc: 38.80%
INFO: trainer: 2025

Training complete. Best Accuracy: 38.8


In [26]:
results

{'training_losses': [2.1351363824473486,
  1.888144251373079,
  1.7981697652075026,
  1.7010747359858618,
  1.6349172592163086],
 'training_accuracies': [23.655555555555555,
  30.244444444444444,
  34.46666666666667,
  37.355555555555554,
  40.37777777777778],
 'validation_losses': [2.3983818888664246,
  1.7181784808635712,
  1.6010252833366394,
  1.772821992635727,
  2.6748618483543396],
 'validation_accuracies': [29.5, 35.2, 38.8, 35.9, 30.9],
 'best_accuracy': 38.8,
 'best_epoch': 3,
 'training_time': 180.59993076324463,
 'cuda_inference_time': 13695.287704467773,
 'total_trainable_params': 11173962,
 'model_memory_usage': 341.1240234375,
 'fitness_val_loss': None,
 'scalar_multi_objective': None,
 'total_flops': 1110845440,
 'confusion_matrix': [[534, 58, 18, 10, 6, 5, 17, 103, 197, 52],
  [38, 491, 3, 10, 1, 1, 8, 93, 20, 335],
  [127, 20, 130, 105, 127, 56, 130, 249, 35, 21],
  [52, 12, 33, 300, 11, 68, 87, 356, 13, 68],
  [59, 14, 60, 88, 211, 30, 182, 325, 11, 20],
  [41, 9, 38